# Phase 2 — Statistical Exploration

**Research question:** *Based on age, stress level, and activity level, can we predict sleep quality?*

**Goal of this notebook:** summarize the key variables with descriptive statistics (mean, median, standard deviation, variance), then measure how strongly each of our three factors actually relates to sleep quality — with correlations and a t-test. This tells us what the models in Phase 4 *should* find, so we can catch them if they lie.

**How the research question maps to dataset columns:**

| Factor in our question | Dataset column(s) | Range |
|---|---|---|
| Age | `age` | 18–69 years |
| Stress level | `stress_score` | 1–10 scale |
| Activity level | `steps_that_day` + `exercise_day` | 500–20,000 steps · 0/1 exercised |
| **Target: sleep quality** | `sleep_quality_score` | 1–10 scale |

Activity gets two columns because the dataset measures it two ways: how much the person moved (steps) and whether they deliberately exercised that day (0/1).

In [1]:
import pandas as pd

df = pd.read_csv("../sleep_health_dataset.csv")

PREDICTORS = ["age", "stress_score", "steps_that_day", "exercise_day"]
TARGET = "sleep_quality_score"

df[PREDICTORS + [TARGET]].head()

,age,stress_score,steps_that_day,exercise_day,sleep_quality_score
0,29,4.4,6592,0,6.6
1,55,4.0,10111,1,6.9
2,42,7.8,9222,1,1.0
3,37,4.9,9190,1,6.4
4,23,7.4,4273,0,3.2


## 1. Descriptive statistics

`describe()` summarizes each column: count, mean, standard deviation, min/max, and quartiles. This is our first quantitative look at *what's typical* in the data.

In [2]:
cols = ["age", "stress_score", "steps_that_day", TARGET]
df[cols].describe().round(2)

,age,stress_score,steps_that_day,sleep_quality_score
count,100000.00,100000.00,100000.00,100000.00
mean,34.71,5.73,7496.86,4.87
std,11.04,1.62,3460.42,1.51
min,18.00,1.00,500.00,1.00
25%,26.00,4.80,5045.00,3.80
50%,33.00,5.80,7442.00,4.90
75%,42.00,6.80,9887.00,6.00
max,69.00,10.00,20000.00,10.00


## 2. Mean, median, SD, and variance side by side

What each measure means:

- **Mean** — the average value.
- **Median** — the middle value when sorted. If mean ≈ median, the distribution isn't badly skewed.
- **Standard deviation (SD)** — how spread out values are around the mean (in the column's own units).
- **Variance** — SD squared. Same information in squared units; we report SD when talking to people because it's interpretable ("±1.5 points"), and mention variance because it's what the math under the models actually uses.

In [3]:
summary = pd.DataFrame({
    "mean": df[cols].mean(),
    "median": df[cols].median(),
    "std": df[cols].std(),
    "variance": df[cols].var(),
})
summary.round(2)

,mean,median,std,variance
age,34.71,33.0,11.04,121.80
stress_score,5.73,5.8,1.62,2.62
steps_that_day,7496.86,7442.0,3460.42,11974533.44
sleep_quality_score,4.87,4.9,1.51,2.27


**Reading the table above:**

- Average sleep quality is about **4.9 / 10**, with SD ≈ 1.5 — so most people score between ~3.4 and ~6.4. There is real spread here for a model to explain.
- Average stress is ~5.7 / 10 with SD ≈ 2.4 — stress varies a lot from person to person.
- Mean ≈ median for every column → no serious skew, no red flags.
- Sanity check on the math: for each column, variance = std² (e.g. 1.5² ≈ 2.25).

## 3. Correlations — which factors actually relate to sleep quality?

Pearson correlation ranges from −1 to +1:

- **−1** — perfect negative relationship (more X → less Y)
- **0** — no linear relationship at all
- **+1** — perfect positive relationship

This one number per predictor is the heart of the whole project: it tells us which of our three factors carries real signal.

In [4]:
correlations = df[PREDICTORS + [TARGET]].corr()[TARGET].drop(TARGET)
correlations.round(3).sort_values()

stress_score     -0.639
age              -0.025
steps_that_day    0.014
exercise_day      0.022
Name: sleep_quality_score, dtype: float64

**Reading the result:** `stress_score` ≈ **−0.64** — a strong negative relationship: higher stress, worse sleep. Every other predictor — `age`, `steps_that_day`, `exercise_day` — sits at ≈ 0, meaning essentially **no linear relationship** with sleep quality in this dataset.

This is our headline finding: **of the three factors in our research question, only stress carries real predictive signal.**

## 4. Inferential statistics — is the stress gap real?

A correlation could in principle be a fluke of the sample. A **t-test** checks whether the difference in average sleep quality between two groups (high-stress vs low-stress people) is *statistically real* or could have happened by chance.

- **t** — how many standard errors apart the two group means are (bigger magnitude = more separated)
- **p-value** — the probability of seeing a gap this large if there were actually no difference. Tiny p (< 0.05) → the gap is real.

In [5]:
from scipy import stats

high_stress = df[df["stress_score"] >= 7][TARGET]
low_stress  = df[df["stress_score"] <= 3][TARGET]

t, p = stats.ttest_ind(high_stress, low_stress)

print(f"high-stress group: n = {len(high_stress):,}, mean sleep quality = {high_stress.mean():.2f}")
print(f"low-stress group:  n = {len(low_stress):,}, mean sleep quality = {low_stress.mean():.2f}")
print(f"gap: {low_stress.mean() - high_stress.mean():.2f} points")
print(f"t = {t:.1f},  p = {p:.2e}")

high-stress group: n = 22,289, mean sleep quality = 3.56
low-stress group:  n = 6,130, mean sleep quality = 6.84
gap: 3.28 points
t = -194.5,  p = 0.00e+00


**Reading the result:** low-stress people sleep about **3.3 points better** (on the 1–10 scale) than high-stress people, and the p-value is effectively zero — this gap is real, not sampling noise.

For contrast, let's run the same test on a factor with ~0 correlation (exercise). Watch both the **gap size** and the p-value — with 100,000 rows, even trivial gaps come out “statistically significant,” so the gap size is what matters practically.

In [6]:
# Contrast: the same t-test on exercise (correlation ~ +0.02)
exercised     = df[df["exercise_day"] == 1][TARGET]
not_exercised = df[df["exercise_day"] == 0][TARGET]

t2, p2 = stats.ttest_ind(exercised, not_exercised)
print(f"exercised:     mean sleep quality = {exercised.mean():.2f}")
print(f"not exercised: mean sleep quality = {not_exercised.mean():.2f}")
print(f"gap: {exercised.mean() - not_exercised.mean():.2f} points   (vs ~2 points for stress)")
print(f"t = {t2:.1f},  p = {p2:.2e}")

exercised:     mean sleep quality = 4.91
not exercised: mean sleep quality = 4.84
gap: 0.07 points   (vs ~2 points for stress)
t = 6.8,  p = 1.04e-11


## 5. Findings

- **Sleep quality averages 4.9 / 10** (SD ≈ 1.5) — there's meaningful spread for a model to predict.
- **Only stress correlates with sleep quality** (−0.64). Age, steps, and exercise all sit at ≈ 0 — essentially no linear signal.
- The t-test confirms the stress effect is **real and large**: high-stress people sleep ~3.3 points worse than low-stress people (p ≈ 0).
- **Significance ≠ importance:** the exercise t-test also comes out “significant” (p ≈ 1e-11) — with n = 100,000, even a 0.07-point gap clears the p < 0.05 bar. But 0.07 points on a 10-point scale is practically nothing next to stress’s 3.3. With big data, always judge the **size** of an effect, not just its p-value.
- **What this means for our research question:** we *can* partly predict sleep quality, but we should expect the models in Phase 4 to lean almost entirely on stress. If a model claims age or steps is important, the numbers here say to be suspicious (see Phase 4's permutation-importance check).
- Caveat: correlations measure *linear* relationships. The Random Forest in Phase 4 can catch non-linear patterns if they exist — comparing both models tests exactly that.

**Next (Phase 3):** turn these numbers into charts — the stress scatter, the flat age/steps scatters, and the correlation heatmap.